# 1 — 声を録る

スマートフォンや PC の録音アプリで自分の声を録音し、音声ファイルを Colab にアップロードして整えます。

録音内容と一致する文字起こしも保存します。自分の声、または使用許可のある声だけを使ってください。

In [ ]:
from pathlib import Path
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from fractions import Fraction
from IPython.display import Audio, display

WORK = Path('/content/voice_clone_data')
WORK.mkdir(parents=True, exist_ok=True)
print('作業領域:', WORK)

## 1. 音声ファイルを指定

Colab Notebookの **Add Input** で自分の録音ファイルを追加し、下の `SOURCE` をそのファイルのパスに変更してください。WAVを推奨します。

In [ ]:
SOURCE = Path('/content/your-voice.wav')  # 実際のパスに変更
if not SOURCE.exists():
    raise FileNotFoundError(f'音声ファイルのパスを確認してください: {SOURCE}')

audio, sr = sf.read(SOURCE, dtype='float32', always_2d=True)
print('サンプルレート:', sr, 'Hz')
print('チャンネル数:', audio.shape[1])
print('長さ:', round(len(audio) / sr, 2), '秒')
display(Audio(audio.T, rate=sr))

## 2. モノラル・32 kHz WAVに変換

元の音声は変更せず、変換した作業用ファイルを作ります。

In [ ]:
mono = audio.mean(axis=1)
target_sr = 32000
if sr != target_sr:
    ratio = Fraction(target_sr, sr).limit_denominator()
    mono = resample_poly(mono, ratio.numerator, ratio.denominator).astype(np.float32)

prepared = WORK / 'voice_mono_32k.wav'
sf.write(prepared, mono, target_sr, subtype='PCM_16')
print('保存:', prepared)
print('長さ:', round(len(mono) / target_sr, 2), '秒')
display(Audio(str(prepared)))

## 3. 参照音声の文字起こし

録音の内容と一致する文字起こしを入力します。次のモデル実行Notebookで参照音声と一緒に使います。

In [ ]:
transcript = 'ここに録音した内容を正確に入力する'
transcript_path = WORK / 'reference_transcript.txt'
transcript_path.write_text(transcript.strip() + '\n', encoding='utf-8')
print('保存:', transcript_path)

## 完了チェック

- [ ] 録音を聞いて、雑音や他人の声が目立たないことを確認
- [ ] 文字起こしを音声と照合
- [ ] WAVと文字起こしを非公開で保管

次は「2 — 声を覚える」で、この参照音声を GPT-SoVITS に渡します。